# 03 · Red objects (g−i > 2) and red-band diagrams

The objects redder than g−i = 2, which are mostly nearby M dwarfs, and colour diagrams in the red bands (i, z, y).

| Code | Figure |
|---|---|
| R0 | g−i > 2 objects over all colour-eligible objects |
| R1, R2, R3 | the same objects by Fink label, by SIMBAD group, and by Gaia parallax / Legacy DR8 pstar |
| D1 | why their faint r limit is brighter: the g-band alert requirement |
| Z1 | i vs i−z, r−i vs i−z, i−z vs z−y, split by programme |
| Z2 | red-band diagrams by SIMBAD group |

**Inputs:** notebook 01's template table and notebook 02's science table, plus the catalog (for the first-alert band).
**Outputs:** `RUN_ROOT/03_red_objects_and_red_bands/`. Run time: about 1 minute.

In [ ]:
# Shared configuration: paths, run folder, helpers (edit notebooks/science/_setup.py, not this cell).
from _setup import *
describe_config()
import red_objects_and_red_bands as red
template_photometry = require(RUN_ROOT / "01_template_colours/tables/dia_object_template_photometry.parquet", "01_template_colours")
science_table = require(RUN_ROOT / "02_science_suite/tables/dia_object_science_table.parquet", "02_science_suite")
out = step_dir("03_red_objects_and_red_bands")
figs = out / "figures"


def new_figures(before):
    show([out / f["file"] for f in record[before:]])

## Step 1 · Prepare the per-object table
Red = (g−i)_tmpl > 2 among objects with usable g, r and i templates. The R3 rows are exclusive, in this order:
Gaia parallax S/N > 5, then Legacy pstar ≥ 0.5, then Legacy pstar < 0.5, then neither. Legacy `pstar` is a star
likelihood from the Legacy colours (Duncan 2022), valid only for PSF-type sources; it is not a morphology flag.

In [ ]:
v = red.prepare(template_photometry, science_table, CATALOG, DATA_ROOT)
record, res = red.start_results(v)
res

## R0 · Overview

In [ ]:
n = len(record); red.fig_r0_overview(v, figs, record, res); new_figures(n)

## R1–R3 · By Fink label, SIMBAD group, and parallax / Legacy pstar

In [ ]:
n = len(record); red.fig_r1_r3_facets(v, figs, record, res); new_figures(n)
res["R1_rows"], res["R2_rows"], res["R3_rows"]

## D1 · Why the faint r limit is brighter for red objects

In [ ]:
n = len(record); red.fig_d1_detection_bands(v, figs, record, res); new_figures(n)
{k: res[k] for k in res if k.startswith("D1_")}

## Z1, Z2 · Red-band diagrams

In [ ]:
n = len(record); red.fig_z1_red_bands(v, figs, record, res); red.fig_z2_red_bands_by_simbad(v, figs, record, res); new_figures(n)
{k: res[k] for k in res if k.startswith("Z1_")}

## Tables and summary

In [ ]:
red.write_tables(v, out, res)
summary = red.build_summary(out, code_version(), template_photometry, science_table, CATALOG, record, res, round(time.time() - T0, 1))
save_json(summary, out / "summary.json")
print(len(record), "figures written to", figs)